# Clase 202 — Drift detection con PSI, K-S, Wasserstein y Evidently

Generamos un dataset de "producción" con shift sintético, detectamos con tests estadísticos manuales y con Evidently.

## Setup

In [ ]:
import numpy as np, pandas as pd
from scipy import stats
from sklearn.datasets import fetch_california_housing

X, y = fetch_california_housing(return_X_y=True, as_frame=True)
rng = np.random.default_rng(42)

# Reference = primer 60%, Current = último 40% con shift
n = len(X)
ref = X.iloc[:int(n * 0.6)].copy()
cur = X.iloc[int(n * 0.6):].copy()

# Shift sintético: 'MedInc' inflación + 'HouseAge' segmento nuevo
cur['MedInc'] = cur['MedInc'] * 1.5
cur = cur[cur['HouseAge'] >= 25]
print('ref:', ref.shape, '| cur:', cur.shape)

## 1. PSI manual

In [ ]:
def psi(ref, cur, bins=10):
    """Population Stability Index entre dos arrays continuos."""
    edges = np.quantile(ref, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    p_ref, _ = np.histogram(ref, bins=edges)
    p_cur, _ = np.histogram(cur, bins=edges)
    p_ref = np.clip(p_ref / p_ref.sum(), 1e-6, 1)
    p_cur = np.clip(p_cur / p_cur.sum(), 1e-6, 1)
    return float(np.sum((p_cur - p_ref) * np.log(p_cur / p_ref)))

print(f'{"feature":15} {"PSI":>8}  interpretación')
for col in X.columns:
    p = psi(ref[col].values, cur[col].values)
    flag = '🟢 estable' if p < 0.1 else '🟡 cambio menor' if p < 0.2 else '🔴 SHIFT'
    print(f'{col:15} {p:>8.4f}  {flag}')

## 2. K-S y Wasserstein (continuas)

In [ ]:
results = []
for col in X.columns:
    ks = stats.ks_2samp(ref[col].values, cur[col].values)
    w = stats.wasserstein_distance(ref[col].values, cur[col].values)
    results.append({'feature': col, 'KS_stat': ks.statistic, 'KS_p': ks.pvalue, 'Wasserstein': w})
pd.DataFrame(results).round(4)

## 3. Sensibilidad de Wasserstein vs K-S a outliers

Si el shift es 'pocos puntos pero muy lejos', K-S puede no detectarlo y Wasserstein sí.

In [ ]:
a = rng.normal(0, 1, 5000)
b = a.copy()
outlier_idx = rng.choice(len(b), 50, replace=False)
b[outlier_idx] = b[outlier_idx] * 50   # 1% de outliers extremos

print('K-S:', stats.ks_2samp(a, b))
print('Wasserstein:', stats.wasserstein_distance(a, b))
print('PSI:', psi(a, b))
print('\n→ K-S puede no ver el cambio (centro intacto); Wasserstein y PSI sí.')

## 4. Reporte Evidently

In [ ]:
# Requiere: pip install evidently
try:
    from evidently.report import Report
    from evidently.metric_preset import DataDriftPreset
    rep = Report(metrics=[DataDriftPreset()])
    rep.run(reference_data=ref, current_data=cur)
    rep.save_html('drift_report.html')
    print('reporte guardado: drift_report.html')
    print('drift detectado en', sum(1 for m in rep.as_dict()['metrics'][0]['result']['drift_by_columns'].values() if m['drift_detected']), 'columnas')
except ImportError:
    print('pip install evidently para esta celda')

## 5. Alerta vía webhook (cooldown)

In [ ]:
import time, json, os
from pathlib import Path

def maybe_alert(webhook_url, message, cooldown_s=14400, marker_path=Path('.last_alert')):
    """Postea a webhook si no hubo alerta reciente. Stub seguro: no hace HTTP si webhook_url es None."""
    last = float(marker_path.read_text()) if marker_path.exists() else 0
    if time.time() - last < cooldown_s:
        print('[skipped] cooldown activo')
        return
    marker_path.write_text(str(time.time()))
    payload = {'text': message}
    if webhook_url:
        import httpx
        httpx.post(webhook_url, json=payload, timeout=5)
    print('[alert]', message)

drift_features = [r['feature'] for r in results if psi(ref[r['feature']], cur[r['feature']]) > 0.2]
if drift_features:
    maybe_alert(None, f'PSI > 0.2 en: {drift_features}. Reporte: s3://.../drift_report.html')
else:
    print('OK — sin drift accionable.')

## Ejercicio guiado

1. Tomá `prediction = model.predict(ref)` y `prediction_cur = model.predict(cur)`. Aplicá PSI sobre las distribuciones de predicción. ¿Coincide con el data drift?
2. Implementá CBPE (NannyML) para estimar `MAE` esperado en `cur` sin usar `y_cur`. Compará con el `MAE` real.
3. Hacé un dashboard Streamlit que toma snapshots diarios y plotea PSI por feature en los últimos 30 días.
4. Escribí un runbook con: (a) qué hacer si PSI dispara en `MedInc`, (b) qué hacer si dispara en TODAS las features (probable bug en pipeline upstream).

## Conclusiones

- PSI es el estándar simple e interpretable; Wasserstein capta lo que K-S no.
- Data drift ≠ concept drift — la única forma de medir el segundo es teniendo labels (o estimarlo con CBPE).
- Alerta sin cooldown + sin runbook = alert fatigue garantizada.
- Reference window fija hasta el siguiente retraining oficial.